In [4]:
# -*- coding: utf-8 -*-
"""
Kurly 카테고리 1페이지 → 각 상품 상세 이동
=> category_name(소분류: 예 '비타민·미네랄'), product_name, brand, price, review_count, review_date, review_text
=> 한 개 CSV 저장
"""

import json, re, time
from urllib.parse import urlparse
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager





# ===== 설정 =====
CATEGORY_URL = "https://www.kurly.com/categories/032009?filters=&page=1"  # 예: 건강식품 > 비타민·미네랄
HEADLESS = False
WINDOW_SIZE = "1920,1080"
SHORT, MID, LONG = 3, 7, 12
SCROLL_STEP = 1200
MAX_PRODUCTS_FROM_PAGE = 1   # 전체는 None
MAX_LOAD_MORE_CLICKS = 25
MAX_REVIEWS_PER_PRODUCT = 1
DATE_PAT = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}")
DIGIT_ONLY = re.compile(r"^[0-9,]+$")

def make_driver() -> uc.Chrome:
    opts = uc.ChromeOptions()
    if HEADLESS: opts.add_argument("--headless=new")
    opts.add_argument("--disable-dev-shm-usage"); opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=ko_KR"); opts.add_argument(f"--window-size={WINDOW_SIZE}")
    opts.add_argument("--start-maximized"); opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    d = uc.Chrome(service=service, options=opts); d.set_page_load_timeout(60); return d

def wait_css(d, css, timeout=MID):  return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))
def wait_xpath(d, xp, timeout=MID): return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.XPATH, xp)))
def safe_click(d, by, v, timeout=MID):
    try:
        el = WebDriverWait(d, timeout).until(EC.element_to_be_clickable((by, v)))
        d.execute_script("arguments[0].scrollIntoView({block:'center'});", el); time.sleep(0.2); el.click(); return True
    except: return False
def slow_scroll_to_bottom(d, step_px=SCROLL_STEP, sleep_sec=0.6, max_loops=30):
    last = 0
    for _ in range(max_loops):
        d.execute_script(f"window.scrollBy(0,{step_px});"); time.sleep(sleep_sec)
        h = d.execute_script("return document.body.scrollHeight;")
        if h == last: break
        last = h

# ========= (수정) 카테고리 소분류명 추출 =========
def collect_links_and_category_name(d, category_url: str) -> Tuple[List[str], str]:
    d.get(category_url); time.sleep(SHORT)

    # URL 경로(/categories/032009)와 일치하는 a[href*='경로']의 텍스트 = 소분류명
    path = urlparse(category_url).path  # '/categories/032009'
    category_name = None
    try:
        # 상단 소분류 네비게이션에서 같은 경로를 가리키는 a 태그
        els = d.find_elements(By.XPATH, f"//div[@id='container']//a[contains(@href,'{path}')][normalize-space()]")
        # 같은 경로를 가리키는 링크가 여러 개인 경우가 드물게 있어 첫 번째 텍스트 사용
        for a in els:
            t = a.text.strip()
            if t:
                category_name = t
                break
    except:
        category_name = None

    # 보강: 실패 시 og:title/title 등
    if not category_name:
        soup = BeautifulSoup(d.page_source, "lxml")
        tag = soup.select_one('meta[property="og:title"]')
        txt = (tag.get("content") if tag else None) or (soup.title.get_text() if soup.title else "")
        category_name = (re.split(r"[|\-–—]", txt)[0].strip() if txt else "UnknownCategory")

    # 상품 링크 수집
    slow_scroll_to_bottom(d, max_loops=25)
    anchors = d.find_elements(By.CSS_SELECTOR, "a[href*='/goods/']")
    links, seen = [], set()
    for a in anchors:
        href = a.get_attribute("href") or ""
        if "/goods/" in href and href not in seen:
            seen.add(href); links.append(href)
    return links, category_name

# ===== JSON-LD(가격/리뷰수) =====
def parse_json_ld_for_price_reviewcount(html: str) -> Dict[str, Any]:
    soup = BeautifulSoup(html, "lxml"); data = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try: j = json.loads(tag.string)
        except: continue
        for obj in (j if isinstance(j, list) else [j]):
            if isinstance(obj, dict) and obj.get("@type") == "Product":
                offers = obj.get("offers"); price = None
                if isinstance(offers, dict): price = offers.get("price")
                elif isinstance(offers, list):
                    for off in offers:
                        if isinstance(off, dict) and off.get("price"): price = off["price"]; break
                if price:
                    try: data["price"] = int(str(price).replace(",", "").strip())
                    except: pass
                ar = obj.get("aggregateRating")
                if isinstance(ar, dict) and ar.get("reviewCount") is not None:
                    try: data["review_count"] = int(ar["reviewCount"])
                    except: pass
                return data
    return data

def extract_price_from_dom(d) -> Optional[int]:
    try:
        title = wait_xpath(d, "//h1", timeout=LONG)
        box = title.find_element(By.XPATH, "./ancestor::section[1] | ./ancestor::div[2]")
        spans = box.find_elements(By.XPATH, ".//span[normalize-space()]")
        cand = []
        for s in spans:
            t = s.text.strip()
            if t and "%" not in t and DIGIT_ONLY.match(t): cand.append(int(t.replace(",", "")))
        if cand: return cand[0]
    except: pass
    return None

def extract_reviewcount_from_dom(d) -> Optional[int]:
    try:
        nodes = d.find_elements(By.XPATH, "//*[contains(.,'후기') and (self::a or self::span or self::div)]")
        for n in nodes:
            m = re.search(r"후기[\s(]*(\d+)", (n.text or "").replace(",", ""))
            if m: return int(m.group(1))
    except: pass
    return None

def open_review_tab(d):
    for xp in ["//nav//li[.//span[contains(.,'후기')]]",
               "//nav//*[self::a or self::button][contains(.,'후기')]",
               "//button[contains(.,'후기')]", "//a[contains(.,'후기')]"]:
        if safe_click(d, By.XPATH, xp, timeout=SHORT): time.sleep(1.0); break
    try:
        WebDriverWait(d, LONG).until(EC.presence_of_element_located(
            (By.XPATH, "//*[contains(@aria-label,'상품 후기') or contains(.,'상품 후기')][self::h2 or self::h3 or self::div]")))
    except: pass

def expand_reviews(d, max_clicks=MAX_LOAD_MORE_CLICKS, max_reviews=MAX_REVIEWS_PER_PRODUCT):
    clicks = 0; prev = 0
    while clicks < max_clicks:
        clicked = False
        for b in d.find_elements(By.XPATH, "//button[contains(.,'더보기') or contains(.,'후기 더보기')]"):
            try:
                if b.is_displayed() and b.is_enabled():
                    d.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.2); b.click(); clicked = True; clicks += 1; time.sleep(1.2)
            except: pass
        for b in d.find_elements(By.XPATH, "//article//button[contains(.,'더보기') or contains(.,'자세히')]"):
            try:
                if b.is_displayed():
                    d.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.15); b.click(); time.sleep(0.2)
            except: pass
        slow_scroll_to_bottom(d, max_loops=2)
        soup = BeautifulSoup(d.page_source, "lxml")
        cur = len(soup.select("article")) + len(soup.select("li"))
        if max_reviews and cur >= max_reviews: break
        if not clicked:
            if cur == prev: break
            prev = cur

def parse_reviews_from_page(d, product_url: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(d.page_source, "lxml")
    root = None
    for sel in ["section[id*='review']", "section[class*='review']", "div[id*='review']", "div[class*='review']"]:
        r = soup.select_one(sel); 
        if r: root = r; break
    if root is None: root = soup
    items = root.select("article") or root.select("li")
    rows = []
    for tag in items:
        ps = [p.get_text(" ", strip=True) for p in tag.select("p") if p.get_text(strip=True)]
        body = max(ps, key=len) if ps else tag.get_text(" ", strip=True)
        if not body or len(body) < 10: continue
        m = DATE_PAT.search(tag.get_text(" ", strip=True))
        date_str = m.group(0) if m else None
        rows.append({"product_url": product_url, "review_date": date_str, "review_text": body})
        if MAX_REVIEWS_PER_PRODUCT and len(rows) >= MAX_REVIEWS_PER_PRODUCT: break
    return rows

def split_brand_from_title(title: str) -> Tuple[Optional[str], str]:
    if not title: return None, ""
    m = re.match(r"\s*\[([^\]]+)\]\s*(.*)", title)
    if m:
        brand = m.group(1).strip(); name = m.group(2).strip()
        return brand, (name if name else title.strip())
    return None, title.strip()

def crawl_one_product(d, url: str) -> Tuple[str, str, Optional[int], Optional[int], List[Dict[str, Any]]]:
    d.get(url); time.sleep(SHORT)
    try: title = wait_xpath(d, "//h1", timeout=LONG).text.strip()
    except:
        try: title = wait_css(d, "h1, h2", timeout=MID).text.strip()
        except: title = ""
    brand, product_name = split_brand_from_title(title)
    j = parse_json_ld_for_price_reviewcount(d.page_source)
    price = j.get("price") or extract_price_from_dom(d)
    review_count = j.get("review_count") or extract_reviewcount_from_dom(d)
    open_review_tab(d); expand_reviews(d)
    reviews = parse_reviews_from_page(d, url)
    return product_name, brand, price, review_count, reviews

def main():
    d = make_driver(); rows = []
    try:
        links, category_name = collect_links_and_category_name(d, CATEGORY_URL)
        if isinstance(MAX_PRODUCTS_FROM_PAGE, int): links = links[:MAX_PRODUCTS_FROM_PAGE]
        print(f"[INFO] 소분류 카테고리: {category_name} | 링크 {len(links)}개")
        for i, link in enumerate(links, 1):
            print(f"[{i}/{len(links)}] {link}")
            product_name, brand, price, review_count, reviews = crawl_one_product(d, link)
            if reviews:
                for r in reviews:
                    rows.append({
                        "category_name": category_name,   # 소분류명
                        "product_name": product_name,
                        "brand": brand,
                        "product_url": link,
                        "price": price,
                        "review_count": review_count,
                        "review_date": r.get("review_date"),
                        "review_text": r.get("review_text"),
                    })
            else:
                rows.append({
                    "category_name": category_name,
                    "product_name": product_name,
                    "brand": brand,
                    "product_url": link,
                    "price": price,
                    "review_count": review_count,
                    "review_date": None,
                    "review_text": None,
                })
    finally:
        d.quit()

    out = f"kurly_health_merged_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    cols = ["category_name","product_name","brand","product_url","price","review_count","review_date","review_text"]
    pd.DataFrame(rows, columns=cols).to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[DONE] 저장: {out} (행 {len(rows)})")

if __name__ == "__main__":
    main()


SessionNotCreatedException: Message: session not created: cannot connect to chrome at 127.0.0.1:61643
from session not created: This version of ChromeDriver only supports Chrome version 142
Current browser version is 141.0.7390.123; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x8258c3
	0x825904
	0x63e76d
	0x67b20d
	0x67a159
	0x67008f
	0x66feb6
	0x6b8993
	0x6b830a
	0x6ac766
	0x67dac0
	0x67ede4
	0xaa7974
	0xaa2bea
	0x84e5b4
	0x83dd28
	0x844d8d
	0x82ded8
	0x82e09c
	0x817d1a
	0x76d35d49
	0x77add6db
	0x77add661
